# SHIPP Stage 6 — OpenRouteService Smoke Test


**Purpose:** prove that Databricks can retrieve the ORS secret and make one real
Matrix API request.

This is **not** the production route-enrichment pipeline.

Production integration must consume eligible Silver listing/request pairs and
preserve the raw response before normalization.


In [0]:
%run ../common/shipp_silver_lib

In [0]:
# Confirm the Databricks secret exists without printing it.

ors_api_key = dbutils.secrets.get(
    scope=ORS_SECRET_SCOPE,
    key=ORS_SECRET_KEY,
)

url = f"{ORS_MATRIX_BASE.rstrip('/')}/{ORS_PROFILE}"

assert ors_api_key

print("ORS secret retrieved successfully.")
print("ORS endpoint:", url)

In [0]:
# Read coordinates from trusted Silver once Stage 5 has passed.

from pyspark.sql import functions as F

assert table_exists(SILVER_LISTINGS), (
    f"BLOCKED: missing {SILVER_LISTINGS}. "
    "Run Stage 5 Silver Listings first."
)

assert table_exists(SILVER_REQUESTS), (
    f"BLOCKED: missing {SILVER_REQUESTS}. "
    "Run Stage 5 Silver Requests first."
)

listing = (
    spark.table(SILVER_LISTINGS)
    .filter(
        F.col("listing_id") == "demo-listing-001"
    )
    .select(
        "longitude",
        "latitude",
    )
    .first()
)

request = (
    spark.table(SILVER_REQUESTS)
    .filter(
        F.col("request_id") == "demo-request-001"
    )
    .select(
        "longitude",
        "latitude",
    )
    .first()
)

assert listing is not None, (
    "demo-listing-001 not found in Silver."
)

assert request is not None, (
    "demo-request-001 not found in Silver."
)

assert listing["longitude"] is not None
assert listing["latitude"] is not None
assert request["longitude"] is not None
assert request["latitude"] is not None

locations = [
    [
        float(listing["longitude"]),
        float(listing["latitude"]),
    ],
    [
        float(request["longitude"]),
        float(request["latitude"]),
    ],
]

print("Silver coordinates loaded.")
print("Listing:", locations[0])
print("Request:", locations[1])

In [0]:
import requests

payload = {
    "locations": locations,
    "sources": [0],
    "destinations": [1],
    "metrics": [
        "distance",
        "duration",
    ],
    "units": "km",
}

headers = {
    "Authorization": ors_api_key,
    "Content-Type": "application/json",
}

print("Calling:", url)

response = requests.post(
    url,
    json=payload,
    headers=headers,
    timeout=ORS_TIMEOUT_SECONDS,
)

print("HTTP status:", response.status_code)

response.raise_for_status()

raw_response = response.json()

assert "distances" in raw_response, (
    "ORS response missing distances."
)

assert "durations" in raw_response, (
    "ORS response missing durations."
)

distance_km = raw_response["distances"][0][0]
duration_seconds = raw_response["durations"][0][0]

assert distance_km is not None
assert duration_seconds is not None

print("Distance (km):", distance_km)
print(
    "Duration (min):",
    round(duration_seconds / 60.0, 1),
)

print(
    "PASS — real ORS Matrix request returned "
    "distance and duration."
)

display(raw_response)


## Evidence to retain

- HTTP status = 200
- ORS endpoint/profile
- redacted request shape — never expose the API key
- raw JSON response
- returned distance
- returned duration
- final PASS line

## Production integration

This notebook proves ORS connectivity only.

The production path is:

`30_silver_candidate_pairs`
→ `31_ors_route_enrichment`
→ `shipp_bronze.ors_route_responses`
→ `32_silver_routes`

Do not persist production route responses from this smoke-test notebook.
